<a href="https://colab.research.google.com/github/A-Kuo/Language-Model-Hallucination-Detection-via-Entropy-Divergence/blob/main/notebooks/pytorch_dl_testbed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch Deep-Learning Testbed — Synthetic Entropy-Divergence Sandbox

**Status: experimental sandbox — not part of the production pipeline.** Local runs write to `notebooks/outputs/` (untracked); Kaggle GPU CI runs (see `notebooks/README.md`) write to `notebooks/results/`, which *is* committed.

This notebook is a **self-contained playground for testing deep-learning ideas in pure PyTorch**
against this project's core research question — *can internal uncertainty signals (token entropy,
divergence between stochastic passes) separate hallucinated from factual generations?* — without
touching any production module in this repo. Ideas that survive here get ported into the real
pipeline afterwards (see the integration checklist at the bottom).

## Research questions under test

| # | Question | Motivation |
|---|----------|------------|
| RQ1 | When do deep probes (MLP / BiLSTM / attention) actually beat a linear probe on entropy signals? | `detector.py`: BiLSTM (~0.78 AUROC) currently *underperforms* LogReg (~0.91). Before investing more in sequence models we want a controlled map of **where** depth pays off. |
| RQ2 | Does multi-pass KL divergence rescue detection of **confident confabulation** (low-entropy hallucinations)? | Root README limitation: *"A model can produce low-entropy hallucinations …"* — single-pass entropy is blind there. The KL signal is the repo's namesake method but was never ablated against this failure mode. |

## How it works — no downloads, no GPUs required

Instead of downloading an LLM, we build a **synthetic pseudo-LM**: a controllable generator of
per-layer next-token logits whose statistics mimic the phenomena the repo measures
(`entropy_baselines.py` features, layer-wise entropy signatures, pass-to-pass instability).
The generative knobs map to real phenomena:

| Knob | Phenomenon |
|------|------------|
| knowledge strength `s` | how grounded the model is in the asked-about entity |
| `gamma` (effect size) | difficulty of the detection problem (how weak the entropy signal is) |
| `confab_frac` | fraction of hallucinations that are *confidently wrong* (low-entropy) |
| per-pass logit noise `sigma(s)` | epistemic instability — larger when knowledge is weaker (MC-dropout analogue) |

All feature extraction reuses the repo's exact conventions (`EPS = 1e-12`, natural-log nats,
the 6D `FEATURE_NAMES` vector from `entropy_baselines.py`, trapezoid AUROC from
`detector.py`) so results transfer conceptually 1:1.

## Logistics

* **Dependencies:** `torch`, `numpy`, `matplotlib` only. Validated with torch 2.11 (CPU) and torch 2.10+cu128 (Kaggle T4 GPU).
* **Runtime:** full-scale run (5 seeds, 8-point gamma sweep, N=2500/dataset) takes tens of minutes on a Kaggle GPU — budget up to an hour; a quick harness smoke test still runs in seconds.
* **Artifacts:** figures + JSON summaries land in `notebooks/outputs/` when run locally (untracked), or `notebooks/results/` when run via the Kaggle GPU workflow (committed).
* **Seeds:** everything is seeded; reruns are reproducible.